# TACO prompts by difficulty

Read only question and difficulty columns through `TacoDataset.rows()`.
Delegate validation, deduplication and JSONL writing to `PromptOperation`.
Each difficulty is deduplicated independently; original statement text is preserved.


## 1. Repository setup

In [1]:
# Standard library: resolve the repository before importing project modules.
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
for directory in (working_directory, *working_directory.parents):
    if (directory / "modeling/src/data/export_prompts.py").is_file():
        PROJECT_ROOT = directory
        break
else:
    raise FileNotFoundError("Open this notebook inside prompt-difficulty-predictor")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODELING_DIR = PROJECT_ROOT / "modeling"
DATA_DIR = MODELING_DIR / "data"


## 2. Imports and configuration

In [2]:
from collections import defaultdict

from modeling.src.data.export_prompts import PromptOperation
from modeling.src.data.taco_dataset import TacoDataset

dataset = TacoDataset(DATA_DIR / "raw/TACO")
prompt_operation = PromptOperation(data_dir=DATA_DIR)
OUTPUT_DIR = DATA_DIR / "processed/por_nivel"


/Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Group existing prompts

In [3]:
prompts_by_difficulty = defaultdict(list)
for split in ("train", "test"):
    for row in dataset.rows(split, columns=["question", "difficulty"]):
        difficulty = str(row["difficulty"] or "UNKNOWN_DIFFICULTY").upper()
        prompts_by_difficulty[difficulty].append(row["question"])

print({difficulty: len(prompts) for difficulty, prompts in sorted(prompts_by_difficulty.items())})


{'EASY': 9104, 'HARD': 3362, 'MEDIUM': 3444, 'MEDIUM_HARD': 2945, 'UNKNOWN_DIFFICULTY': 5014, 'VERY_HARD': 2574}


## 4. Export through the existing prompt operation

In [4]:
for difficulty, prompts in sorted(prompts_by_difficulty.items()):
    filename = "".join(
        character if character in "abcdefghijklmnopqrstuvwxyz0123456789_" else "_"
        for character in difficulty.lower()
    )
    summary = prompt_operation.export_prompts(prompts, OUTPUT_DIR / f"{filename}.jsonl")
    print(difficulty, summary)


EASY {'read': 9104, 'exported': 9074, 'empty_or_short': 0, 'duplicates': 30}
HARD {'read': 3362, 'exported': 3189, 'empty_or_short': 0, 'duplicates': 173}
MEDIUM {'read': 3444, 'exported': 3416, 'empty_or_short': 0, 'duplicates': 28}
MEDIUM_HARD {'read': 2945, 'exported': 2884, 'empty_or_short': 0, 'duplicates': 61}
UNKNOWN_DIFFICULTY {'read': 5014, 'exported': 4880, 'empty_or_short': 0, 'duplicates': 134}
VERY_HARD {'read': 2574, 'exported': 2395, 'empty_or_short': 0, 'duplicates': 179}
